In [ ]:
# Run once in the project root, then pick the .venv kernel for this notebook:
# python3 -m venv .venv && .venv/bin/pip install -r requirements.txt

SyntaxError: invalid syntax (3848228664.py, line 2)

In [7]:
import cv2
import dlib
import numpy as np
import matplotlib.pyplot as plt
import time
import os

In [8]:
# dlib's 68-point landmark model; download_models.py fetches it if missing.
import subprocess, sys
if not os.path.exists("shape_predictor_68_face_landmarks.dat"):
    subprocess.run([sys.executable, "download_models.py"], check=True)

In [9]:
print(os.path.exists("shape_predictor_68_face_landmarks.dat"))

True


In [10]:
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

if face_detector.empty():
    raise RuntimeError("Face detector could not be loaded")

print("Face detector loaded")

Face detector loaded


In [11]:
landmark_predictor = dlib.shape_predictor(
    "shape_predictor_68_face_landmarks.dat"
)

print("68-point landmark predictor loaded")

68-point landmark predictor loaded


In [19]:
IMAGE_PATH = "/Users/arpanbhattacharya/driver_drowsiness/data/test2.jpg"   # your own test image (not in git), placed next to this notebook

In [20]:
image_path = IMAGE_PATH

image = cv2.imread(image_path)

if image is None:
    raise RuntimeError(f"Could not read image: {image_path}")

print(image.shape)

(960, 1440, 3)


In [21]:
gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)

faces = face_detector.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(80, 80)
)

print("Faces detected:", len(faces))

Faces detected: 1


In [22]:
image_display = image.copy()

for (x, y, w, h) in faces:

    cv2.rectangle(
        image_display,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        2
    )

plt.figure(figsize=(10, 7))

plt.imshow(
    cv2.cvtColor(
        image_display,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")
plt.show()

In [23]:
landmarks = None

if len(faces) > 0:

    face = max(
        faces,
        key=lambda rect: rect[2] * rect[3]
    )

    x, y, w, h = face

    dlib_face = dlib.rectangle(
        int(x),
        int(y),
        int(x + w),
        int(y + h)
    )

    shape = landmark_predictor(
        gray,
        dlib_face
    )

    landmarks = np.array([
        (point.x, point.y)
        for point in shape.parts()
    ])

if landmarks is None:
    raise RuntimeError("No face found in the image")

print("Landmark shape:", landmarks.shape)

Landmark shape: (68, 2)


In [24]:
landmark_image = image.copy()

for i, (x, y) in enumerate(landmarks):

    cv2.circle(
        landmark_image,
        (int(x), int(y)),
        2,
        (0, 255, 0),
        -1
    )

    cv2.putText(
        landmark_image,
        str(i),
        (int(x) + 3, int(y) - 3),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.3,
        (255, 0, 0),
        1
    )

plt.figure(figsize=(12, 8))

plt.imshow(
    cv2.cvtColor(
        landmark_image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")
plt.show()

In [25]:
LEFT_EYE_START = 36
LEFT_EYE_END = 42

RIGHT_EYE_START = 42
RIGHT_EYE_END = 48

left_eye = landmarks[
    LEFT_EYE_START:LEFT_EYE_END
]

right_eye = landmarks[
    RIGHT_EYE_START:RIGHT_EYE_END
]

print("Left eye:")
print(left_eye)

print("\nRight eye:")
print(right_eye)

Left eye:
[[629 460]
 [653 436]
 [689 433]
 [723 456]
 [691 468]
 [655 472]]

Right eye:
[[850 453]
 [884 427]
 [920 428]
 [945 450]
 [922 462]
 [885 462]]


In [26]:
eye_image = image.copy()

for eye in [left_eye, right_eye]:

    for x, y in eye:

        cv2.circle(
            eye_image,
            (int(x), int(y)),
            4,
            (0, 255, 0),
            -1
        )

plt.figure(figsize=(12, 8))

plt.imshow(
    cv2.cvtColor(
        eye_image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")
plt.show()

In [27]:
def euclidean_distance(p1, p2):

    p1 = np.asarray(p1)
    p2 = np.asarray(p2)

    return np.linalg.norm(p1 - p2)

In [28]:
p1 = np.array([0, 0])
p2 = np.array([3, 4])

print(euclidean_distance(p1, p2))

5.0


In [29]:
def calculate_ear(eye):

    vertical_1 = euclidean_distance(
        eye[1],
        eye[5]
    )

    vertical_2 = euclidean_distance(
        eye[2],
        eye[4]
    )

    horizontal = euclidean_distance(
        eye[0],
        eye[3]
    )

    ear = (
        vertical_1 + vertical_2
    ) / (
        2.0 * horizontal
    )

    return ear

In [30]:
left_ear = calculate_ear(left_eye)
right_ear = calculate_ear(right_eye)

ear = (
    left_ear +
    right_ear
) / 2.0

print(f"Left EAR : {left_ear:.3f}")
print(f"Right EAR: {right_ear:.3f}")
print(f"Average EAR: {ear:.3f}")

Left EAR : 0.378
Right EAR: 0.363
Average EAR: 0.371


## Per-frame pipeline and channel confidence
Steps 1-4 (face -> landmarks -> eye points -> EAR) packed into one function so the **video file and the live camera share exactly the same code**.

**Confidence** (0-1) says how much to trust the EAR of this frame. It is the *minimum* of five simple factors (a chain is only as strong as its weakest link), so any one problem lowers it:

| factor | drops when |
|---|---|
| `size` | eyes are only a few pixels wide (far away / low resolution) |
| `pose` | head is turned: one eye looks narrower than the other |
| `inside` | eye points fall outside the image (face cut off) |
| `light` | eye region is too dark or over-exposed |
| `stable` | eye landmarks jump between frames (tracking is jittery) |

No face -> confidence 0. Frames below `MIN_CONF` are labelled `UNKNOWN`, so they are never counted as "open" or "closed".

In [31]:

EAR_THRESHOLD = 0.20      # fallback only; replaced by the adaptive value below
MIN_CONF = 0.3            # below this a frame is "UNKNOWN", not OPEN/CLOSED


def calibrate_threshold(open_ears, fallback, min_samples=30):
    """Adaptive closed-eye threshold = 0.7 x the user's own median open-eye EAR.
    Open-eye EAR differs between people (small eyes ~0.22, wide eyes ~0.35),
    so a fixed value is too strict for some and too loose for others."""
    if len(open_ears) < min_samples:
        return fallback
    return float(np.clip(0.7 * np.median(open_ears), 0.12, 0.25))


def clip01(x):
    return float(np.clip(x, 0.0, 1.0))


def eye_confidence(gray, left_eye, right_eye, prev_eyes=None):
    """Return (confidence, factors). Each factor is in [0,1]; confidence = min of them."""
    height, width = gray.shape
    eyes = np.vstack([left_eye, right_eye]).astype(float)

    width_l = euclidean_distance(left_eye[0], left_eye[3])
    width_r = euclidean_distance(right_eye[0], right_eye[3])

    # size: EAR is noisy when the eye is only a few pixels wide (10 px -> 0, 25 px -> 1)
    size = clip01(((width_l + width_r) / 2 - 10) / 15)

    # pose: when the head turns, the far eye looks narrower (width ratio 0.5 -> 0, 0.8 -> 1)
    pose = clip01((min(width_l, width_r) / max(width_l, width_r, 1e-6) - 0.5) / 0.3)

    # inside: share of the 12 eye points that are inside the image
    inside_share = np.mean(
        (eyes[:, 0] >= 0) & (eyes[:, 0] < width) & (eyes[:, 1] >= 0) & (eyes[:, 1] < height)
    )
    inside = clip01((inside_share - 0.5) / 0.5)

    # light: mean brightness of the box around the eyes (dark < 35, blown out > 240)
    x0, y0 = np.maximum(eyes.min(axis=0).astype(int), 0)
    x1, y1 = eyes.max(axis=0).astype(int) + 1
    region = gray[y0:y1, x0:x1]
    if region.size == 0:
        light = 0.0
    else:
        brightness = region.mean()
        light = min(clip01((brightness - 35) / 35), clip01((240 - brightness) / 30))

    # stable: average landmark shift since last frame, relative to the distance between the eyes
    stable = 1.0
    if prev_eyes is not None:
        interocular = euclidean_distance(left_eye.mean(axis=0), right_eye.mean(axis=0))
        shift = np.linalg.norm(eyes - prev_eyes, axis=1).mean() / max(interocular, 1e-6)
        stable = 1.0 - clip01((shift - 0.05) / 0.25)

    factors = dict(size=size, pose=pose, inside=inside, light=light, stable=stable)
    return min(factors.values()), factors


def process_frame(frame, threshold, prev_eyes=None):
    """Steps 1-4 on one BGR frame. Never raises on a missing face."""
    result = dict(
        face_box=None, eyes=None, left_ear=np.nan, right_ear=np.nan, ear=np.nan,
        state="UNKNOWN", confidence=0.0, factors={},
    )

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 1. detect face (largest one if several)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))
    if len(faces) == 0:
        return result
    x, y, w, h = max(faces, key=lambda rect: rect[2] * rect[3])

    # 2. landmarks, 3. keep only the eye points
    shape = landmark_predictor(gray, dlib.rectangle(int(x), int(y), int(x + w), int(y + h)))
    landmarks = np.array([(p.x, p.y) for p in shape.parts()])
    left_eye, right_eye = landmarks[36:42], landmarks[42:48]

    # 4. EAR
    left_ear, right_ear = calculate_ear(left_eye), calculate_ear(right_eye)
    ear = (left_ear + right_ear) / 2.0

    confidence, factors = eye_confidence(gray, left_eye, right_eye, prev_eyes)
    if confidence < MIN_CONF:
        state = "UNKNOWN"
    else:
        state = "CLOSED" if ear < threshold else "OPEN"

    result.update(
        face_box=(x, y, w, h), eyes=np.vstack([left_eye, right_eye]).astype(float),
        left_ear=left_ear, right_ear=right_ear, ear=ear,
        state=state, confidence=confidence, factors=factors,
    )
    return result


In [32]:
# The test image shows open eyes, so it gives this person's adaptive threshold
res = process_frame(image, EAR_THRESHOLD)
if res["confidence"] >= MIN_CONF:
    EAR_THRESHOLD = calibrate_threshold([res["ear"]], EAR_THRESHOLD, min_samples=1)
    res = process_frame(image, EAR_THRESHOLD)

print(f"Adaptive EAR threshold: {EAR_THRESHOLD:.3f}")
print(f"EAR {res['ear']:.3f}  state {res['state']}  confidence {res['confidence']:.2f}")
print({k: round(v, 2) for k, v in res["factors"].items()})

EAR 0.371  state OPEN  confidence 1.00
{'size': 1.0, 'pose': 1.0, 'inside': 1.0, 'light': 1.0, 'stable': 1.0}


In [33]:
VIDEO_PATH = "/Users/arpanbhattacharya/driver_drowsiness/data/test_vid_1.mov"   # your own test video (not in git), placed next to this notebook

In [34]:
video_path = VIDEO_PATH

print("Video:", video_path)

Video: /Users/arpanbhattacharya/driver_drowsiness/data/test_vid_1.mov


In [35]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Could not open video")

fps = cap.get(cv2.CAP_PROP_FPS)

frame_count = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

duration = frame_count / fps if fps > 0 else 0

print("FPS:", fps)
print("Frames:", frame_count)
print("Resolution:", width, "x", height)
print("Duration:", duration, "seconds")

FPS: 29.678215911721583
Frames: 641
Resolution: 1620 x 1080
Duration: 21.598333333333333 seconds


In [36]:
ear_history = []
time_history = []
eye_state_history = []     # "OPEN" / "CLOSED" / "UNKNOWN" (no face or low confidence)
conf_history = []
frame_history = []

In [37]:
print(f"EAR threshold in use: {EAR_THRESHOLD:.3f}")

In [38]:
frame_number = 0
prev_eyes = None

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # every frame is logged, also those without a face, so time stays correct
    res = process_frame(frame, EAR_THRESHOLD, prev_eyes)
    prev_eyes = res["eyes"]

    ear_history.append(res["ear"])
    time_history.append(frame_number / fps)
    eye_state_history.append(res["state"])
    conf_history.append(res["confidence"])
    frame_history.append(frame_number)

    frame_number += 1


cap.release()

known = sum(s != "UNKNOWN" for s in eye_state_history)

print("Processing complete")
print("Frames processed:", len(ear_history))
print("Frames with a usable face:", known)

Processing complete
Frames processed: 641
Frames with a usable face: 527


In [39]:
plt.figure(figsize=(15, 5))

plt.plot(
    time_history,
    ear_history
)

plt.axhline(
    EAR_THRESHOLD,
    linestyle="--",
    label="EAR threshold"
)

plt.xlabel("Time (seconds)")
plt.ylabel("EAR")

plt.title(
    "Eye Aspect Ratio over Time"
)

plt.legend()

plt.grid()

plt.show()

In [40]:
def find_closures(times, states):
    """List of eye closures. Duration is measured in seconds from timestamps,
    so it is right for a video file and for a camera with a varying frame rate."""
    closures = []
    start = None
    dt = float(np.median(np.diff(times))) if len(times) > 1 else 0.0

    for t, state in zip(times, states):

        if state == "CLOSED" and start is None:
            start = t

        elif state != "CLOSED" and start is not None:
            closures.append({"start_time": start, "end_time": t, "duration": t - start})
            start = None

    if start is not None:                       # still closed at the end
        end = times[-1] + dt
        closures.append({"start_time": start, "end_time": end, "duration": end - start})

    return closures


closures = find_closures(time_history, eye_state_history)

In [41]:
print(
    "Number of eye closures:",
    len(closures)
)

for i, closure in enumerate(closures):

    print(
        f"Closure {i + 1}: "
        f"{closure['duration']:.3f} seconds"
    )

Number of eye closures: 17
Closure 1: 0.101 seconds
Closure 2: 0.034 seconds
Closure 3: 0.034 seconds
Closure 4: 0.034 seconds
Closure 5: 0.067 seconds
Closure 6: 0.775 seconds
Closure 7: 0.168 seconds
Closure 8: 0.168 seconds
Closure 9: 0.573 seconds
Closure 10: 0.607 seconds
Closure 11: 0.640 seconds
Closure 12: 0.034 seconds
Closure 13: 0.101 seconds
Closure 14: 0.034 seconds
Closure 15: 0.034 seconds
Closure 16: 0.034 seconds
Closure 17: 0.034 seconds


In [42]:
BLINK_MAX_DURATION = 0.5
DROWSY_CLOSURE_DURATION = 1.5
MIN_CLOSURE_DURATION = 0.06    # shorter than ~2 frames is landmark noise, not a blink

In [43]:
closures = [c for c in closures if c["duration"] >= MIN_CLOSURE_DURATION]

blinks = []
prolonged_closures = []

for closure in closures:

    duration = closure["duration"]

    if duration <= BLINK_MAX_DURATION:

        blinks.append(closure)

    elif duration >= DROWSY_CLOSURE_DURATION:

        prolonged_closures.append(closure)

In [44]:
print("Blinks:", len(blinks))

print(
    "Prolonged closures:",
    len(prolonged_closures)
)

Blinks: 5
Prolonged closures: 0


In [45]:
video_duration = time_history[-1]

blink_count = len(blinks)

blink_frequency = (
    blink_count / video_duration
) * 60

print(
    f"Blink frequency: "
    f"{blink_frequency:.2f} blinks/minute"
)

Blink frequency: 13.91 blinks/minute


In [46]:
# PERCLOS counts only frames where the state is known: a lost face is not "alert"
closed_frames = sum(state == "CLOSED" for state in eye_state_history)
total_frames = sum(state != "UNKNOWN" for state in eye_state_history)

perclos = closed_frames / total_frames if total_frames else float("nan")

print(f"PERCLOS: {perclos:.3f}")
print(f"PERCLOS percentage: {perclos * 100:.2f}%")

PERCLOS: 0.195
PERCLOS percentage: 19.54%


In [47]:
def detect_drowsiness(
    perclos,
    prolonged_closures
):

    if len(prolonged_closures) > 0:

        return "DROWSY"

    if perclos > 0.30:

        return "DROWSY"

    return "ALERT"

In [48]:
drowsiness_state = detect_drowsiness(
    perclos,
    prolonged_closures
)

print(
    "Final state:",
    drowsiness_state
)

Final state: ALERT


In [49]:
def calculate_drowsiness_score(perclos, longest_closure, blink_frequency, observed_seconds):
    """Score in [0,1], 0 = alert, 1 = drowsy. Built only from window-level values,
    so one normal blink cannot move it."""

    score = 0.0

    if not np.isnan(perclos):
        score += 0.50 * min(perclos / 0.30, 1.0)               # PERCLOS >= 30 % -> full

    score += 0.35 * min(longest_closure / DROWSY_CLOSURE_DURATION, 1.0)

    if observed_seconds >= 20 and blink_frequency < 5:         # blink rate needs a long window
        score += 0.15

    return min(score, 1.0)

In [50]:
longest_closure = max([c["duration"] for c in closures], default=0.0)

drowsiness_score = calculate_drowsiness_score(
    perclos,
    longest_closure,
    blink_frequency,
    video_duration
)

print(
    f"Drowsiness score: "
    f"{drowsiness_score:.2f}"
)

Drowsiness score: 0.51


In [51]:
if drowsiness_score >= 0.7:

    print("🚨 DROWSY")

elif drowsiness_score >= 0.4:

    print("⚠️ WARNING")

else:

    print("✓ ALERT")

⚠️ WARNING


In [52]:
plt.figure(figsize=(15, 5))

plt.plot(
    time_history,
    ear_history,
    label="EAR"
)

plt.axhline(
    EAR_THRESHOLD,
    linestyle="--",
    label="Threshold"
)

for closure in prolonged_closures:

    start = closure["start_time"]

    end = closure["end_time"]

    plt.axvspan(
        start,
        end,
        alpha=0.3
    )

plt.xlabel("Time (seconds)")
plt.ylabel("EAR")

plt.title(
    "EAR and Prolonged Eye Closures"
)

plt.legend()
plt.grid()

plt.show()

## Channel output (the contract used by fusion later)
`channel_output` turns a window of per-frame results into **score, confidence and raw values**. The confidence of the channel is the mean frame confidence over the window, so lost faces (confidence 0) pull it down.

In [53]:

def channel_output(times, states, confs, window_s=30.0):
    """Summarise the last `window_s` seconds. Used for the video file and the live camera."""
    times = np.asarray(times, dtype=float)
    keep = times >= times[-1] - window_s
    t = times[keep]
    s = [st for st, k in zip(states, keep) if k]
    c = np.asarray(confs, dtype=float)[keep]

    closures_all = find_closures(t, s)
    current_closure = closures_all[-1]["duration"] if s[-1] == "CLOSED" else 0.0
    closures_ok = [x for x in closures_all if x["duration"] >= MIN_CLOSURE_DURATION]
    blinks_n = sum(x["duration"] <= BLINK_MAX_DURATION for x in closures_ok)
    longest = max([x["duration"] for x in closures_ok], default=0.0)

    n_known = sum(st != "UNKNOWN" for st in s)
    perclos = sum(st == "CLOSED" for st in s) / n_known if n_known else float("nan")
    observed = float(t[-1] - t[0])
    blink_rate = blinks_n / observed * 60 if observed > 0 else 0.0

    return {
        "score": calculate_drowsiness_score(perclos, longest, blink_rate, observed * n_known / len(s)),
        "confidence": float(c.mean()),
        "raw": {
            "eyes_closed": s[-1] == "CLOSED",
            "current_closure_s": current_closure,
            "longest_closure_s": longest,
            "blink_count": blinks_n,
            "blink_rate_per_min": blink_rate,
            "perclos": perclos,
            "face_found_share": n_known / len(s),
            "window_s": observed,
        },
        "timestamp": float(t[-1]),
    }


output = channel_output(time_history, eye_state_history, conf_history, window_s=float("inf"))

print(f"score      : {output['score']:.2f}   (0 = alert, 1 = drowsy)")
print(f"confidence : {output['confidence']:.2f}")
for key, value in output["raw"].items():
    print(f"  {key}: {value}")


score      : 0.51   (0 = alert, 1 = drowsy)
confidence : 0.79
  eyes_closed: False
  current_closure_s: 0.0
  longest_closure_s: 0.7749791991679658
  blink_count: 5
  blink_rate_per_min: 13.911663708619495
  perclos: 0.1954459203036053
  face_found_share: 0.8221528861154446
  window_s: 21.56463858554342


## Live detection (laptop camera, other camera index, video file or stream URL)
`run_live(source)` uses the same `process_frame` and `channel_output` as above, on a rolling window (default 30 s).

* **Calibration:** the first `calibrate_s` seconds (eyes open, look at the camera) set the closed-eye threshold to 0.7 x your median open-eye EAR (clipped to 0.12-0.25). People's open-eye EAR differs (about 0.25-0.40), so one fixed 0.20 is too strict for some and too loose for others. Falls back to 0.20 if too few good frames.
* **macOS camera:** the app that runs this notebook (Terminal, VS Code, Jupyter) needs permission under System Settings > Privacy & Security > Camera.
* **Stop:** interrupt the kernel (notebook), or press `q` (with `show="window"`), or set `max_seconds`.

In [54]:

from collections import deque
from IPython.display import display, Image


def draw_hud(frame, res, out, threshold, note=""):
    """Draw face box, eye points and the main values on the frame (in place)."""
    color = {"OPEN": (0, 200, 0), "CLOSED": (0, 0, 255), "UNKNOWN": (0, 165, 255)}[res["state"]]
    if res["face_box"] is not None:
        x, y, w, h = res["face_box"]
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        for px, py in res["eyes"]:
            cv2.circle(frame, (int(px), int(py)), 2, (0, 255, 0), -1)

    lines = [f"{res['state']}  EAR {res['ear']:.2f} (thr {threshold:.2f})"]
    lines.append(f"CONFIDENCE frame {res['confidence']:.2f}" + (f"  channel {out['confidence']:.2f}" if out else ""))
    if out is not None:
        r = out["raw"]
        lines += [
            f"PERCLOS {r['perclos']:.2f}  blinks {r['blink_count']} ({r['blink_rate_per_min']:.0f}/min)",
            f"closed now {r['current_closure_s']:.1f}s  longest {r['longest_closure_s']:.1f}s",
            f"SCORE {out['score']:.2f}",
        ]
    if res["factors"]:
        weakest = min(res["factors"], key=res["factors"].get)
        lines.append(f"weakest factor: {weakest} {res['factors'][weakest]:.2f}")
    if note:
        lines.append(note)
    for i, text in enumerate(lines):
        cv2.putText(frame, text, (10, 22 + 22 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 3)
        cv2.putText(frame, text, (10, 22 + 22 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    # prolonged closure: red banner across the bottom
    if out is not None and out["raw"]["current_closure_s"] >= DROWSY_CLOSURE_DURATION:
        height, width = frame.shape[:2]
        cv2.rectangle(frame, (0, height - 60), (width, height), (0, 0, 255), -1)
        cv2.putText(frame, f"ALERT! EYES CLOSED {out['raw']['current_closure_s']:.1f}s",
                    (10, height - 20), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)


def run_live(source=0, window_s=30.0, calibrate_s=10.0, max_seconds=None, show="notebook"):
    """source: camera index (0 = laptop camera), video path or stream URL.
    show: "notebook" (inline image), "window" (OpenCV window, q to quit) or None.
    Returns the last channel output."""
    if isinstance(source, str) and source.isdigit():
        source = int(source)
    is_camera = isinstance(source, int)

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open source {source!r}. For a camera: allow camera access for this app in "
            "System Settings > Privacy & Security > Camera, and check the index (0, 1, ...)."
        )

    threshold = EAR_THRESHOLD       # starts from the image-based value
    calibrated = calibrate_s <= 0
    recent_ears, last_state = deque(maxlen=3), "UNKNOWN"
    open_ears = []
    times, states, confs = deque(), deque(), deque()
    prev_eyes, output, handle = None, None, None
    black_frames, fps_smooth = 0, 0.0
    t_start = time.time()

    try:
        while True:
            t_loop = time.perf_counter()
            ok, frame = cap.read()
            if not ok:
                print("Stream ended or camera stopped delivering frames.")
                break

            # a permission problem on macOS usually shows up as all-black frames
            black_frames = black_frames + 1 if frame.mean() < 2 else 0
            if black_frames == 30:
                print("Camera frames are black: macOS camera permission may be missing for this app.")

            t = time.time() - t_start if is_camera else cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
            if max_seconds is not None and t > max_seconds:
                break

            if not calibrated and t >= calibrate_s:
                threshold = calibrate_threshold(open_ears, threshold)
                calibrated = True
                print(f"Adaptive EAR threshold: {threshold:.3f}")

            res = process_frame(frame, threshold, prev_eyes)
            prev_eyes = res["eyes"]

            # steadiness: average EAR over 3 frames, and a 15 % margin before an
            # eye already CLOSED is called OPEN again (stops flicker around the threshold)
            if res["state"] != "UNKNOWN":
                if not calibrated:
                    open_ears.append(res["ear"])
                recent_ears.append(res["ear"])
                res["ear"] = float(np.mean(recent_ears))
                margin = 1.15 if last_state == "CLOSED" else 1.0
                res["state"] = "CLOSED" if res["ear"] < threshold * margin else "OPEN"
            else:
                recent_ears.clear()
            last_state = res["state"]

            note = ""
            if not calibrated:
                note = f"CALIBRATING {calibrate_s - t:.0f}s: eyes open, look at the camera"
                res["state"] = "UNKNOWN"        # no OPEN/CLOSED verdict until the threshold is known
            else:
                times.append(t); states.append(res["state"]); confs.append(res["confidence"])
                while t - times[0] > window_s:
                    times.popleft(); states.popleft(); confs.popleft()
                output = channel_output(times, states, confs, window_s)

            latency = time.perf_counter() - t_loop
            fps_smooth = 1 / latency if fps_smooth == 0 else 0.9 * fps_smooth + 0.1 / latency
            draw_hud(frame, res, output, threshold, note + f"  {fps_smooth:.0f} FPS, {latency * 1000:.0f} ms")

            if show == "window":
                cv2.imshow("EAR live (q to quit)", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
            elif show == "notebook":
                image_bytes = Image(data=cv2.imencode(".jpg", frame)[1].tobytes())
                if handle is None:
                    handle = display(image_bytes, display_id=True)
                else:
                    handle.update(image_bytes)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        cap.release()
        if show == "window":
            cv2.destroyAllWindows()

    return output


In [56]:

# 0 = laptop camera; try 1 for another camera, or a video path / stream URL
live_output = run_live(source=0, window_s=30, calibrate_s=10)
print(live_output)


Calibrated EAR threshold: 0.218
Stopped.
{'score': 0.85, 'confidence': 0.5092338184748731, 'raw': {'eyes_closed': True, 'current_closure_s': np.float64(0.06667816638946533), 'longest_closure_s': np.float64(1.7337288856506348), 'blink_count': np.int64(32), 'blink_rate_per_min': np.float64(64.06039222055094), 'perclos': 0.4666666666666667, 'face_found_share': 0.6094808126410836, 'window_s': 29.971717834472656}, 'timestamp': 83.45771384239197}
